<a href="https://colab.research.google.com/github/rutendomuzendah/Lab-tickets-/blob/main/BudgetBuddyMainProgram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gradio as gr
import hands_on_ai
from hands_on_ai.chat import get_response
from hands_on_ai.chat import register

warnings.filterwarnings('ignore')

os.environ['HANDS_ON_AI_SERVER'] = 'https://ollama.serveur.au'
os.environ['HANDS_ON_AI_MODEL'] = 'llama3.2'
os.environ['HANDS_ON_AI_API_KEY'] = 'isys2001smartfinane

def load_and_clean_data(file_obj):
    if file_obj is None:
        return None

    df = pd.read_csv(file_obj.name)

    required_columns = ["Date", "Amount", "Category", "Description"]
    for col in required_columns:
        if col not in df.columns:
            raise gr.Error(f"Spreadsheet Structure Alert: Missing mandatory column '{col}'")

    df["Amount"] = df["Amount"].astype(str).str.replace(r'[\$,]', '', regex=True)
    df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")


    df["Category"] = df["Category"].astype(str).str.strip()

    df = df.dropna(subset=["Amount", "Date"])
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)
    df = df.dropna(subset=["Date"])

    df = df.drop_duplicates(subset=["Date", "Amount", "Description"])
    return df


def generate_financial_metrics(df, monthly_income, budgets):
    refunds = df[df["Amount"] < 0]
    spending = df[df["Amount"] > 0]

    gross_spending = spending["Amount"].sum()
    net_spending = gross_spending + refunds["Amount"].sum()
    avg_transaction = spending["Amount"].mean() if len(spending) > 0 else 0

    essentials = df[df["Category"].isin(["Groceries", "Transport", "Utilities"])]["Amount"].sum()
    non_essentials = df[df["Category"].isin(["Coffee", "Dining", "Entertainment", "Shopping"])]["Amount"].sum()
    non_essential_flag = non_essentials > (0.5 * gross_spending)

    actual_savings_realized = monthly_income - net_spending

    alerts = []
    for category, limit in budgets.items():
        spent = df[df["Category"] == category]["Amount"].sum()
        if spent > limit:
            alerts.append(f"🚨 Budget Blown in {category}: Spent ${spent:,.2f} of Allowed ${limit:,.2f}")
        elif spent > 0.8 * limit:
            alerts.append(f"⚠️ Warning for {category}: Spent ${spent:,.2f} (Approaching ${limit:,.2f} cap)")

    return {
        "gross": gross_spending,
        "net": net_spending,
        "avg": avg_transaction,
        "essentials": essentials,
        "non_essentials": non_essentials,
        "non_essential_flag": non_essential_flag,
        "savings": actual_savings_realized,
        "alerts": alerts
    }

def fetch_macro_financial_news(focus_topic):
    """
    Simulates a live high-level economic news API stream, tracking geopolitical,
    inflationary, and supply-chain pressures affecting student living costs.
    """
    news_database = {
        "Global Inflation & Geopolitics": (
            "📰 [GLOBAL ECONOMIC MONITOR]\n"
            "• CRUDE OIL SPIKE: Global Brent crude rose 14% this quarter due to trade route changes and ongoing conflict corridors. Local fuel prices are hitting record highs, impacting public transit fares and delivery fees.\n"
            "• SUPPLY-CHAIN PRESSURES: Major maritime disruptions have added critical delays to consumer goods shipping lanes, forcing international retailers to push logistics cost overheads directly onto clothing and electronics retail pricing arrays."
        ),
        "Domestic Food & Energy Crises": (
            "📰 [DOMESTIC MARKET BRIEFING]\n"
            "• AGRICULTURAL DEARNESS: Severe climate patterns and a 30% surge in commercial fertilizer shipping costs have restricted agricultural yield outputs. Core grocery staples (grains, fresh produce, dairy) have outpaced base inflation matrices by 8.4%.\n"
            "• UTILITY GRID CORRECTION: Regional grid overhauls have introduced updated seasonal pricing tariffs. Household and campus residential power bills have advanced by an average of 11.2%, hitting student grocery and housing budgets heavily."
        )
    }
    return news_database.get(focus_topic, "No active economic updates selected.")

def calculate_savings_forecast(target_amount: float, monthly_savings: float) -> str:
    if monthly_savings <= 0:
        return "⚠️ Target Forecast: Your current cash velocity is flat or negative. To start building savings, reduce lifestyle categories (Coffee/Shopping) first."

    months_needed = target_amount / monthly_savings
    return f"🎯 Target Progress: At your current savings speed of ${monthly_savings:,.2f}/month, you will successfully clear your target of ${target_amount:,.2f} in exactly {months_needed:.1f} months

def create_plots(df, metrics):
    if df is None or len(df) == 0:
        return None, None

    fig1, ax1 = plt.subplots(figsize=(5, 5))
    df_positive = df[df["Amount"] > 0]
    category_totals = df_positive.groupby("Category")["Amount"].sum()
    if not category_totals.empty:
        category_totals.plot(kind=\"pie\", autopct=\"%1.1f%%\", ax=ax1, title=\"Spending Proportions Matrix\")
        ax1.set_ylabel("")
    plt.tight_layout()
    plot1_path = "category_pie.png"
    fig1.savefig(plot1_path)
    plt.close(fig1)

    fig2, ax2 = plt.subplots(figsize=(7, 3.5))
    df_sorted = df.sort_values("Date")
    df_sorted.set_index("Date").resample("W")["Amount"].sum().plot(kind="bar", ax=ax2, color="teal", title="Weekly Spending Velocity")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plot2_path = "spending_trend.png"
    fig2.savefig(plot2_path)
    plt.close(fig2)

    return plot1_path, plot2_path


def process_pipeline(file_obj, income, target_goal, active_news_topic):
    if file_obj is None:
        return "### 🏦 System Notification\nPlease upload your transactional banking CSV file export to launch the Budget Buddy analysis pipeline.", None, None

    student_budgets = {"Groceries": 400, "Transport": 150, "Dining": 200, "Entertainment": 150}

    try:
        df = load_and_clean_data(file_obj)
        metrics = generate_financial_metrics(df, float(income), student_budgets)
        pie_img, trend_img = create_plots(df, metrics)
    except Exception as e:
        return f"### ❌ Data Pipeline Parsing Failure\nSystem error logs: {str(e)}", None, None

    # Fetch selected real-world macro context variables
    selected_news_context = fetch_macro_financial_news(active_news_topic)

    # Step 10 RAG Construction: Merging structural banking math with the real-world economic news stream
    system_context_prompt = f"""
    You are 'Budget Buddy', a helpful, witty, and highly clear personal financial mentor for university students.
    A student has provided their banking transactional spreadsheet, alongside an active macro-economic news updates panel.

    Speak directly to them about their behaviors using these exact financial metrics:
    - Gross Spending Outflows: ${metrics['gross']:,.2f}
    - Net Final Position (Adjusted for Returns/Refunds): ${metrics['net']:,.2f}
    - Mean Average Transaction Size: ${metrics['avg']:,.2f}
    - Total Essential Outlays (Groceries + Transport + Utilities): ${metrics['essentials']:,.2f}
    - Total Lifestyle Cost Outlays (Coffee + Dining + Shopping): ${metrics['non_essentials']:,.2f}
    - Realized Net Period Savings: ${metrics['savings']:,.2f}

    System Boundary Monitoring Report logs: {', '.join(metrics['alerts']) if metrics['alerts'] else 'Excellent. All monitored parameters are safely inside expected budget boundaries.'}
    Did lifestyle leisure categories skip past 50% of total gross outflows? {metrics['non_essential_flag']}.

    CURRENT MACROECONOMIC BACKGROUND INTELLIGENCE CONTEXT:
    {selected_news_context}

    TASK DIRECTION:
    Review their transaction metrics, alert records, and explicitly connect their spending lines back to the global/domestic news context provided.
    (For example: If they spent heavily on transport or groceries, emphasize how rising oil shipping costs or crop shortages detailed in the news feed are driving those exact totals up).
    Provide a witty, supportive analysis and give them exactly 3 highly specific, practical tips on how they can protect their savings goal during this specific economic climate.
    """

    goal_forecast_string = calculate_savings_forecast(float(target_goal), metrics['savings'])

    try:
        ai_response = get_response(system_context_prompt)
        report_output = f"### 🏦 Budget Buddy Contextual Analysis & Mentorship Advice\n\n{ai_response}\n\n🎬 **System Goal Calculations:**\n{goal_forecast_string}"
    except Exception as network_timeout:
        report_output = f"### 📊 Processed Transaction Summary\n\n*Notice: LLM service timeout. Pure backend analytical reporting parsed below.*\n\n{selected_news_context}\n\n- **Gross Spending Outlays:** ${metrics['gross']:,.2f}\n- **Calculated Period Net Position:** ${metrics['net']:,.2f}\n- **Calculated Net Available Savings:** ${metrics['savings']:,.2f}\n\n**Goal Calculation:** {goal_forecast_string}"

    return report_output, pie_img, trend_img

GRAPHICAL WEB APPLICATION DESIGN LAYOUT (GRADIO UI)

with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🏦 Budget Buddy: Smart Student Finance Assistant (Context-Aware Edition)")
    gr.Markdown("Upload your banking CSV file and select a live global economic headline to see how macroeconomic forces impact your day-to-day student budget.")

    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(label="Upload CSV Transaction Ledger", file_types=[".csv"])
            income_input = gr.Number(label="Expected Monthly Income Allowance ($)", value=2000)
            goal_input = gr.Number(label="Target Financial Savings Goal ($)", value=1000)

            # Interactive drop-down to feed the API simulation stream
            news_selector = gr.Dropdown(
                label="Select Live Macro-Economic News Feed Channel",
                choices=["Global Inflation & Geopolitics", "Domestic Food & Energy Crises"],
                value="Global Inflation & Geopolitics"
            )

            submit_btn = gr.Button("Generate Insights", variant="primary")

        with gr.Column(scale=2):
            ai_report = gr.Markdown(label="Budget Buddy Interactive Report Block")
            with gr.Row():
                chart_out1 = gr.Image(label="Expenditure Proportions Matrix")
                chart_out2 = gr.Image(label="Weekly Velocity Trends")

    submit_btn.click(
        fn=process_pipeline,
        inputs=[file_input, income_input, goal_input, news_selector],
        outputs=[ai_report, chart_out1, chart_out2]
    )

if __name__ == "__main__":
    app.launch()